# 01 - Gold: Empresas (dimensiones y agregados de RUES + CIIU + TRM)

Trabajo Práctico 2 - Transformación Silver → Gold

## Objetivo

Construir, a partir de `silver.empresas_rues`, `silver.actividades_ciiu` y `silver.tasa_cambio` (ya validadas y libres de registros rechazados), las tablas analíticas:

* `gold.dim_empresa` — dimensión de empresas (pass-through tipado de `silver.empresas_rues`).
* `gold.dim_actividad_economica` — dimensión del catálogo CIIU (pass-through de `silver.actividades_ciiu`).
* `gold.fact_actividad_economica_resumen` — empresas activas/canceladas y tasa de cancelación por actividad económica.
* `gold.fact_rues_mensual` — nuevas matrículas y cancelaciones por mes, cruzadas con la TRM promedio mensual.

## Preguntas de negocio que responden

| Tabla | Pregunta |
|---|---|
| `dim_empresa` | ¿Cómo se distribuyen las empresas por tipo de organización jurídica y por cámara de comercio? (agregación directa por `organizacion_juridica`, `camara_comercio`) |
| `fact_actividad_economica_resumen` | ¿Qué actividades económicas tienen más empresas activas y cuáles más cancelaciones? (tasa de cancelación por CIIU) |
| `fact_rues_mensual` | ¿Cuántas empresas nuevas se matriculan cada mes? ¿Influye la TRM en el cierre de empresas? |

Probado localmente con pandas contra una muestra real de Bronze (`test_gold_local.py`) antes de escribir este notebook: la lógica de agrupación, el join con el catálogo CIIU y el cruce mensual con TRM corren sin error sobre datos reales.

In [ ]:
from pyspark.sql import functions as F

df_empresas = spark.table("Datos_Empresas.silver.empresas_rues")
df_ciiu = spark.table("Datos_Empresas.silver.actividades_ciiu")
df_trm = spark.table("Datos_Empresas.silver.tasa_cambio")

print(f"silver.empresas_rues: {df_empresas.count():,} filas")
print(f"silver.actividades_ciiu: {df_ciiu.count():,} filas")
print(f"silver.tasa_cambio: {df_trm.count():,} filas")

---

## `gold.dim_empresa`

Pass-through tipado de `silver.empresas_rues` (sin lógica adicional: Silver ya validó y limpió). Responde directamente, por agregación, la pregunta de organización jurídica x cámara de comercio.

In [ ]:
df_dim_empresa = df_empresas.withColumn("_processed_at", F.current_timestamp())

df_dim_empresa.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "Datos_Empresas.gold.dim_empresa"
)
print(f"Escrito: Datos_Empresas.gold.dim_empresa ({df_dim_empresa.count():,} filas)")

---

## `gold.dim_actividad_economica`

Pass-through del catálogo CIIU validado en Silver. Es la dimensión que se usa para traducir `cod_ciiu_act_econ_pri` a una descripción legible.

In [ ]:
df_dim_actividad = df_ciiu.withColumn("_processed_at", F.current_timestamp())

df_dim_actividad.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "Datos_Empresas.gold.dim_actividad_economica"
)
print(f"Escrito: Datos_Empresas.gold.dim_actividad_economica ({df_dim_actividad.count():,} filas)")

---

## `gold.fact_actividad_economica_resumen`

Grano: una fila por código CIIU (`cod_ciiu_act_econ_pri`). Agrega `dim_empresa` por actividad económica y calcula `tasa_cancelacion = empresas_canceladas / total_empresas`. Se enlaza con `dim_actividad_economica` para traer la descripción.

Nota: el join contra el catálogo CIIU no cubre el 100% de los códigos (499 de 603 códigos distintos de RUES tienen coincidencia, ~83%, verificado en vivo) — los que no coinciden quedan con `description = NULL`, visible y trazable, no se descartan filas.

In [ ]:
df_resumen_actividad = (
    df_empresas
    .withColumn("_activa", F.when(F.col("estado_matricula") == "ACTIVA", 1).otherwise(0))
    .withColumn("_cancelada", F.when(F.col("estado_matricula").contains("CANCELADA"), 1).otherwise(0))
    .groupBy("cod_ciiu_act_econ_pri")
    .agg(
        F.count("*").alias("total_empresas"),
        F.sum("_activa").alias("empresas_activas"),
        F.sum("_cancelada").alias("empresas_canceladas"),
    )
    .withColumn("tasa_cancelacion", F.round(F.col("empresas_canceladas") / F.col("total_empresas"), 4))
    .join(
        df_ciiu.select(F.col("code").alias("cod_ciiu_act_econ_pri"), F.col("description").alias("descripcion_actividad")),
        on="cod_ciiu_act_econ_pri",
        how="left",
    )
    .withColumn("_processed_at", F.current_timestamp())
)

df_resumen_actividad.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "Datos_Empresas.gold.fact_actividad_economica_resumen"
)
print(f"Escrito: Datos_Empresas.gold.fact_actividad_economica_resumen ({df_resumen_actividad.count():,} filas)")

---

## `gold.fact_rues_mensual`

Grano: un mes (`AAAA-MM`). Combina nuevas matrículas (`fecha_matricula`) y cancelaciones (`fecha_cancelacion`) por mes con la TRM promedio de ese mismo mes (`silver.tasa_cambio`, truncada a mes).

**Advertencia de alcance** (ya señalada por el usuario): la TRM solo cubre ~13 meses (2025-09 a 2026-10), así que el cruce con cancelaciones es un análisis exploratorio, no una conclusión estadísticamente sólida — para los meses de RUES anteriores a esa ventana, `trm_promedio` queda `NULL`.

In [ ]:
df_nuevas_mes = (
    df_empresas.filter(F.col("fecha_matricula").isNotNull())
    .withColumn("mes", F.date_trunc("month", F.col("fecha_matricula")))
    .groupBy("mes")
    .agg(F.count("*").alias("nuevas_matriculas"))
)

df_canceladas_mes = (
    df_empresas.filter(F.col("fecha_cancelacion").isNotNull())
    .withColumn("mes", F.date_trunc("month", F.col("fecha_cancelacion")))
    .groupBy("mes")
    .agg(F.count("*").alias("cancelaciones"))
)

df_trm_mensual = (
    df_trm.withColumn("mes", F.date_trunc("month", F.col("vigenciadesde")))
    .groupBy("mes")
    .agg(F.round(F.avg("valor"), 2).alias("trm_promedio"))
)

df_rues_mensual = (
    df_nuevas_mes.join(df_canceladas_mes, on="mes", how="outer")
    .join(df_trm_mensual, on="mes", how="left")
    .withColumn("_processed_at", F.current_timestamp())
    .orderBy("mes")
)

df_rues_mensual.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "Datos_Empresas.gold.fact_rues_mensual"
)
print(f"Escrito: Datos_Empresas.gold.fact_rues_mensual ({df_rues_mensual.count():,} filas)")

---

## Validación

In [ ]:
%sql
-- Q7: distribucion por organizacion juridica y camara (top 5)
SELECT organizacion_juridica, camara_comercio, COUNT(*) AS empresas
FROM Datos_Empresas.gold.dim_empresa
GROUP BY organizacion_juridica, camara_comercio
ORDER BY empresas DESC
LIMIT 5;

In [ ]:
%sql
-- Q6: actividades con mas empresas activas y mayor tasa de cancelacion
SELECT cod_ciiu_act_econ_pri, descripcion_actividad, total_empresas, empresas_activas, empresas_canceladas, tasa_cancelacion
FROM Datos_Empresas.gold.fact_actividad_economica_resumen
ORDER BY total_empresas DESC
LIMIT 10;

In [ ]:
%sql
-- Q8/Q9: nuevas matriculas, cancelaciones y TRM promedio por mes (ventana TRM)
SELECT mes, nuevas_matriculas, cancelaciones, trm_promedio
FROM Datos_Empresas.gold.fact_rues_mensual
WHERE trm_promedio IS NOT NULL
ORDER BY mes;

---

## Siguiente paso

Continuar con `02_Gold_Contratos.ipynb` (misma carpeta `Trabajo Practico 2 - Capa Gold`; depende de que `gold.dim_empresa` ya exista, se usa para el join con SECOP).